# 03 — Model Walkthrough

Step through the VQA model architecture:
- Instantiate VQAModel and inspect submodules
- Forward-pass trace with a dummy sample
- Layer-by-layer tensor shapes
- Parameter counts per component
- GradCAM saliency map placeholder

In [ ]:
import os, sys
os.chdir('..')
sys.path.insert(0, '.')

In [ ]:
import yaml
import torch
from pathlib import Path

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load vocab and instantiate model

In [ ]:
from src.data.answer_vocab import AnswerVocab
from src.models.vqa_model import VQAModel

vocab_path = Path(cfg['paths']['vocab_path'])
if not vocab_path.exists():
    raise FileNotFoundError('Run python scripts/build_vocab.py first.')
vocab = AnswerVocab.load(vocab_path)
print(f'Vocab size: {len(vocab):,}')

model = VQAModel(
    vision_backbone=cfg['model']['vision_backbone'],
    text_encoder=cfg['model']['text_encoder'],
    hidden_dim=cfg['model']['hidden_dim'],
    num_heads=cfg['model']['num_heads'],
    fusion_layers=cfg['model']['fusion_layers'],
    num_answer_classes=len(vocab),
    dropout=cfg['model']['dropout'],
).to(device)
print(model)

## 2. Parameter counts per component

In [ ]:
def count_params(module):
    return sum(p.numel() for p in module.parameters())

components = {
    'VisionEncoder': model.vision_encoder,
    'TextEncoder':   model.text_encoder,
    'Fusion':        model.fusion,
    'Classifier':    model.classifier,
}
total = count_params(model)
for name, mod in components.items():
    n = count_params(mod)
    print(f'{name:20s}  {n/1e6:7.2f}M  ({100*n/total:.1f}%)')
print(f'{"TOTAL":20s}  {total/1e6:7.2f}M')

## 3. Forward-pass trace with dummy inputs

In [ ]:
from transformers import BertTokenizer
from src.data.augmentations import get_val_transforms

tokenizer = BertTokenizer.from_pretrained(cfg['model']['text_encoder'])
transform = get_val_transforms(cfg['data']['image_size'])

# Dummy batch (B=2)
B = 2
dummy_image = torch.zeros(B, 3, cfg['data']['image_size'], cfg['data']['image_size']).to(device)
enc = tokenizer(
    ['What color is the car?', 'How many people are there?'],
    padding='max_length', truncation=True,
    max_length=cfg['data']['max_question_length'], return_tensors='pt',
)
input_ids = enc['input_ids'].to(device)
attention_mask = enc['attention_mask'].to(device)

model.eval()
with torch.no_grad():
    logits = model(dummy_image, input_ids, attention_mask)

print(f'Input image shape  : {dummy_image.shape}')
print(f'Input ids shape    : {input_ids.shape}')
print(f'Output logits shape: {logits.shape}  (B x num_answers)')

## 4. Mode comparison (multimodal / text_only / image_only)

In [ ]:
for mode in ['multimodal', 'text_only', 'image_only']:
    m = VQAModel(
        vision_backbone=cfg['model']['vision_backbone'],
        text_encoder=cfg['model']['text_encoder'],
        hidden_dim=cfg['model']['hidden_dim'],
        num_heads=cfg['model']['num_heads'],
        fusion_layers=cfg['model']['fusion_layers'],
        num_answer_classes=len(vocab),
        mode=mode,
    ).to(device).eval()
    with torch.no_grad():
        out = m(dummy_image, input_ids, attention_mask)
    print(f'mode={mode:12s}  logits shape: {out.shape}')

## 5. GradCAM saliency placeholder

Run `src/utils/gradcam.py` on a real sample after training to visualise
which image regions the model attends to for a given question.

In [ ]:
from src.utils.gradcam import GradCAM
# Usage (after training):
# cam = GradCAM(model, target_layer=model.vision_encoder.model.vision_model.encoder.layers[-1])
# saliency = cam(pixel_values, input_ids, attention_mask, class_idx=predicted_idx)
print('GradCAM ready — run after loading a trained checkpoint.')